# DermSAM: Closing the Deployment Gap in Skin Lesion Segmentation

This notebook presents the results of a benchmark study evaluating SAM and MedSAM on ISIC 2018 melanoma segmentation, with a focus on the **deployment gap**.

---

## The Core Argument

Published SAM benchmarks consistently report performance using prompts derived from ground-truth masks — centroids, tight bounding boxes — information that simply does not exist at clinical deployment time. A radiologist or dermatologist using SAM in practice cannot provide a ground-truth bounding box; they haven't seen the mask yet.

This creates an **optimistic illusion**: models look better in papers than they will ever perform in the clinic.

**This project quantifies that gap and builds a pipeline to close it.**

The two-stage approach:
1. A lightweight EfficientNet-B0 localizer predicts a bounding box from the image alone — no ground truth used
2. MedSAM uses that auto-generated box as its prompt

The result is a fully automatic, deployable pipeline.

In [ ]:
import sys
sys.path.insert(0, '..')

from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

FIGURES = Path('../outputs/figures')
METRICS = Path('../outputs/metrics')

def show_figure(path, figsize=(14, 6)):
    fig, ax = plt.subplots(figsize=figsize)
    ax.imshow(mpimg.imread(path))
    ax.axis('off')
    plt.tight_layout()
    plt.show()

---

## 1. Benchmark Results

Seven approaches evaluated on the ISIC 2018 test set (260 images).

- **Blue** — supervised baseline (UNet)
- **Red** — unrealistic upper bounds (GT-derived prompts)
- **Green** — realistic deployable approaches (auto-generated prompts)

The gap between red and green rows is the deployment gap this project addresses.

In [ ]:
df = pd.read_csv(METRICS / 'benchmark.csv')

def style_row(row):
    if 'UNREALISTIC' in row['Approach']:
        return ['background-color: #fde8e8'] * len(row)
    elif 'REALISTIC' in row['Approach']:
        return ['background-color: #e8f5e9'] * len(row)
    return ['background-color: #e3f2fd'] * len(row)

display_df = df[['Approach', 'Dice mean', 'Dice std', 'IoU mean', 'IoU std', 'HD95 mean', 'HD95 std']].copy()
display_df['Dice'] = display_df.apply(lambda r: f"{r['Dice mean']:.4f} ± {r['Dice std']:.4f}", axis=1)
display_df['IoU'] = display_df.apply(lambda r: f"{r['IoU mean']:.4f} ± {r['IoU std']:.4f}", axis=1)
display_df['HD95'] = display_df.apply(lambda r: f"{r['HD95 mean']:.1f} ± {r['HD95 std']:.1f}", axis=1)
display_df = display_df[['Approach', 'Dice', 'IoU', 'HD95']]

display_df.style.apply(style_row, axis=1)

In [ ]:
show_figure(FIGURES / 'results_table.png', figsize=(16, 4))

---

## 2. The Deployment Gap

The bar chart below makes the argument visually explicit.

The red bars are what gets reported in papers. The green bars are what you actually get in deployment. The gap between them is the contribution of this work — and the auto-prompt pipeline (green) is what closes it.

The dotted line marks the published ResUNet++ baseline (Dice 0.7726, Jha et al. 2019) — the benchmark to beat.

In [ ]:
show_figure(FIGURES / 'deployment_gap.png', figsize=(12, 6))

---

## 3. Prompt Sensitivity Analysis

How robust is MedSAM to imperfect prompts?

We take the GT bounding box and artificially degrade it — expanding each side by N pixels — then measure Dice at each perturbation level. This gives a **tolerance curve** showing how quickly performance degrades as the prompt gets sloppier.

The red dot marks where the auto-prompt (localizer bbox) actually lands on this curve. This answers the question: *how imperfect is a realistic prompt, relative to the GT upper bound?*

In [ ]:
sensitivity_df = pd.read_csv(METRICS / 'prompt_sensitivity.csv')
print(sensitivity_df.to_string(index=False))

In [ ]:
show_figure(FIGURES / 'prompt_sensitivity.png', figsize=(10, 6))

---

## 4. Qualitative Results

Five columns per sample:
- **Image** — original dermoscopy image
- **GT Mask** — ground truth segmentation
- **UNet** — supervised baseline prediction
- **MedSAM + GT bbox** — unrealistic upper bound
- **MedSAM + Auto bbox** — realistic deployable prediction

Dice scores shown per prediction. The auto-bbox column is the one that matters for real-world use.

In [ ]:
show_figure(FIGURES / 'qualitative_grid.png', figsize=(16, 20))

---

## 5. Failure Case Analysis

The six worst-performing cases for the auto-prompt pipeline, annotated by failure type.

Understanding failure modes is as important as reporting mean performance. The pipeline fails in predictable ways:
- **Poor localizer bbox** — localizer mislocalizes the lesion, passing a bad prompt to MedSAM
- **Small lesions** — localizer bbox IoU degrades on tiny lesions
- **Hair artefacts** — confound the localizer's spatial attention
- **Ambiguous boundaries / atypical morphology** — hard even with a perfect prompt

Yellow rectangle = auto-generated bbox. Red overlay = predicted segmentation.

In [ ]:
show_figure(FIGURES / 'failure_cases.png', figsize=(14, 10))

---

## 6. Summary

| Finding | Value |
|---|---|
| Deployment gap (MedSAM zero-shot GT vs auto) | ~0.07 Dice |
| Auto-prompt pipeline vs published baseline | beats 0.7726 |
| Localizer bbox IoU | 0.693 |
| GradCAM bbox as prompt | insufficient (Dice 0.429) |

**Key takeaways:**

1. MedSAM with a realistic auto-prompt still beats the published ResUNet++ baseline — the deployment gap is real but recoverable with a lightweight localizer
2. GradCAM activations are too diffuse for reliable prompting — a dedicated bbox regression head is necessary
3. The prompt sensitivity curve shows MedSAM is relatively robust to moderate bbox imprecision (up to ~50px), but degrades sharply beyond that
4. Fine-tuning MedSAM on ISIC data further closes the gap — quantified in rows 6–7 of the benchmark table

---

**Dataset:** ISIC 2018 Task 1 — 2594 dermoscopy images, 80/10/10 split, seed 42  
**Hardware:** Google Colab T4 GPU  
**Code:** [github.com/theomalaper/dermSAM](https://github.com/theomalaper/dermSAM)